# Can the rest of a chart tell you a patient is on dialysis?

Pick one diagnosis code and try to predict it from every *other* code on the
same discharge record.

The code here is **Z99.2 — "dependence on renal dialysis"**. It is a good test
case: common enough to model (2.37% of discharges), clinically well defined,
and — as it turns out — surrounded by codes that give the answer away, which
makes it a useful lesson in how a medical prediction model can score
brilliantly while learning nothing.

Prerequisite: `python src/build_transactions.py`.

In [ ]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd

import predict_dialysis as pdz

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 62)

X, codes, lookup, y = pdz.load()
print(f"{X.shape[0]:,} discharges, {y.sum():,} on dialysis "
      f"({100 * y.mean():.2f}%)")

## 1. Decide how to score it before fitting anything

Dialysis dependence appears on 2.37% of records. That single fact rules out
accuracy as a metric: a "model" that answers *no* to everyone, always, is
correct 97.6% of the time and has learned nothing whatsoever.

So the model trains on a class-weighted objective, but is **evaluated at the
true 2.37% prevalence** using ROC-AUC and average precision. Average
precision has a meaningful floor here — random guessing scores 0.0235.

In [ ]:
prevalence = y.mean()
pd.Series({
    "prevalence of Z99.2": f"{100 * prevalence:.2f}%",
    "accuracy of answering 'no' to everyone": f"{100 * (1 - prevalence):.2f}%",
    "average precision of random guessing": f"{prevalence:.4f}",
}).to_frame("")

In [ ]:
Xs, ys = pdz.sample(X, y)
print(f"modelling on a {Xs.shape[0]:,}-row subsample "
      f"at the natural class balance ({100 * ys.mean():.2f}% positive)")

## 2. First attempt: give the model everything

The obvious starting point — every code on the record except the target
itself. Feature count and regularisation strength are chosen by
cross-validated grid search, and a random forest is fit alongside for
comparison.

In [ ]:
all_regimes = pdz.regimes(codes, lookup)
name_full = "all codes available"

rows_full, coefs_full = pdz.run_regime(
    name_full, all_regimes[name_full], Xs, ys, codes, lookup
)
pd.DataFrame(rows_full)[["model", "roc_auc", "avg_precision", "ap_baseline"]]

An average precision of ~0.85 against a 0.024 floor, and ROC-AUC ~0.996.

That is a suspiciously good result for a problem this messy, so the next step
is not to celebrate it — it is to look at *which* features are carrying it.

In [ ]:
coefs_full.head(12)[["code", "description", "odds_ratio"]]

## 3. The problem

Read those descriptions carefully.

`Z91.15` is **"patient's noncompliance with renal dialysis"** — a code that
cannot be assigned to someone who is not on dialysis. `N18.6` is end-stage
renal disease, and ICD-10-CM *instructs* the coder to record `Z99.2`
alongside it when the patient is on dialysis. `I12.0` and `I13.2`
("hypertensive chronic kidney disease with stage 5 CKD or ESRD") carry the
same instruction.

The model is not diagnosing anyone. It is noticing that the chart already
says "dialysis" a few columns over — the equivalent of predicting who owns a
car by checking whether they hold car insurance.

This is **target leakage**, and it is the most common way a medical
prediction model produces a number that cannot survive contact with reality:
in deployment those codes are assigned *by the same process* that assigns the
label, so they would not be available at prediction time.

In [ ]:
leaks = pdz.leak_terms(lookup)
present = [c for c in codes if c in leaks]
print(f"{len(present):,} codes in this dataset name dialysis or ESRD "
      f"in their own description")
lookup.loc[present[:10], ["description"]]

## 4. Remove the giveaways and refit

Rather than hand-listing suspects, the filter is derived from the code
descriptions themselves, so nothing is missed for want of thinking of it.
Two progressively stricter regimes:

- **no dialysis/CKD-definitional codes** — drop anything whose description
  names dialysis or ESRD, every `N18*`/`I12*`/`I13*` code, and the named
  downstream consequences of kidney failure.
- **no renal category at all** — additionally drop every code in AHRQ's
  kidney-related categories, plus anything whose description mentions the
  organ. This is the honest question: *from the non-kidney parts of the
  chart alone, can you still tell?*

In [ ]:
for nm, mask in all_regimes.items():
    print(f"{nm:<40} {mask.sum():>6,} features")

In [ ]:
rows_all = list(rows_full)
coefs = {name_full: coefs_full}

for nm in ["no dialysis/CKD-definitional codes", "also no renal category at all"]:
    r, c = pdz.run_regime(nm, all_regimes[nm], Xs, ys, codes, lookup)
    rows_all += r
    coefs[nm] = c

In [ ]:
comparison = pd.DataFrame(rows_all)[
    ["regime", "model", "roc_auc", "avg_precision", "ap_baseline", "recall"]
]
comparison.style.format({"roc_auc": "{:.3f}", "avg_precision": "{:.3f}",
                         "ap_baseline": "{:.4f}", "recall": "{:.3f}"})

## 5. What the drop means, and what survives

Average precision falls by more than half once the definitional codes are
gone. That gap *is* the leakage — it was never medical signal.

The interesting part is what is left. With every renal and dialysis code
removed, ROC-AUC is still ~0.95: the rest of the chart genuinely identifies
these patients. The surviving predictors reconstruct the systemic signature
of end-stage kidney disease without ever naming the organ.

In [ ]:
coefs["also no renal category at all"].head(20)[
    ["code", "description", "odds_ratio"]
]

Three clinical stories in that list, none of which mention kidneys:

- **Mineral and bone disease** — disorders of mineral metabolism, calcium
  metabolism, hyperparathyroidism, unspecified bone disorders. Failing
  kidneys stop regulating calcium and phosphate, and the skeleton pays for
  it.
- **Fluid overload** — kidneys that cannot excrete water.
- **Vascular access failure** — haemorrhage, infection, thrombosis and
  stenosis of implanted vascular devices. These are the surgically created
  fistulas and grafts that dialysis needles go into, three times a week.

One honest caveat: that last group is dialysis access. It sits in a
*cardiovascular* device category rather than a renal one, so a
category-based filter does not catch it. Some residual leakage remains, and
claiming a perfectly clean model here would be overstating the result.

## 6. How much does each part of the signature carry?

Coefficients answer "what is the effect of this code holding the others
fixed", which understates a group of correlated codes — the five vascular
access complications move together, so each one's individual coefficient
looks small.

Grouped **permutation importance** answers the question actually being asked.
Shuffle a group's values across patients, leaving everything else intact, and
measure how far the score falls. `src/feature_importance.py` does this on the
held-out set with 5 repeats.

In [ ]:
import feature_importance as fi

fi.main()

The named groups are the *strongest and most interpretable* part of the
signal, not the whole of it: together they account for roughly a third of the
score, while the ~3,560 other codes the model uses account for most of the
rest. A clean four-bullet clinical story would overstate how tidy this is.

## 7. Is this the best condition to try?

Dialysis dependence is an administrative status code — the model reaches it
indirectly, through the wreckage around it. `src/compare_targets.py` runs the
same model against six other conditions, scored as "how many times better than
chance" so that codes with very different prevalences stay comparable.

Dialysis keeps its stricter, condition-specific clue filter here rather than
the automatic one used for the others. Under the automatic filter it would
score roughly twice as high — by counting kidney codes the generic rule failed
to recognise as giveaways. The conservative number is the honest one.

In [ ]:
import compare_targets as ct

ct.main()

**Severe sepsis with septic shock** turns out to be the better showcase:
it beats dialysis, and the explanation is causal rather than circumstantial —
severe infections plus the organ failures they trigger.

The bottom of the list is as informative as the top. **Anaemia** manages only
twice chance, which is a fact about the condition rather than a failure of the
model: it appears in almost every kind of patient for almost every reason, so
the rest of the chart barely narrows it down. **Smoking** sits low for a
different reason — it is a behaviour, not a physiological state, so the body
does not necessarily record it.

## 8. Save

`src/predict_dialysis.py` runs all three regimes end to end and writes the
comparison table and the de-leaked predictor list to `results/`.

In [ ]:
from config import RESULTS

out = pd.DataFrame(rows_all)[
    ["regime", "model", "roc_auc", "avg_precision", "ap_baseline",
     "accuracy", "accuracy_of_always_no", "recall", "precision", "brier"]
]
out.to_csv(RESULTS / "dialysis_model_comparison.csv", index=False)
coefs["also no renal category at all"].head(20).to_csv(
    RESULTS / "dialysis_top_predictors_deleaked.csv", index=False)
print(f"wrote 2 files to {RESULTS}")